# ARCHELEC CORPUS PREPROCESSING - LEGISLATIVE ELECTIONS (1973-1993) 

This notebook implements preprocessing of the Archelec corpus (directly available in https://gitlab.teklia.com/ckermorvant/arkindex_archelec/-/tree/master/text_files?ref_type=heads). It is restricted to legislative elections spanning from 1973 to 1993. It cleans the text, removes stopwords, implements lemmatization and finds n-grams.    

# Set Up the Environment

In [1]:
pip install --upgrade gensim scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
!python -m spacy download fr_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 16.9 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')


In [3]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from gensim.models import Phrases
from gensim.models.phrases import Phraser
import re
import unicodedata


# Define paths relative to this notebook's location (script/ folder)
current_dir = Path.cwd()
project_root = current_dir.parent
meta_dir = project_root / "metadata"
data_dir = project_root / "data"
# spacy_model_path = project_root / "fr_core_news_md-3.8.0-py3-none-any.whl"
nlp = spacy.load("fr_core_news_md")

The code below reads the metadata file and merges the texts to each candidate. 

In [4]:
meta_file = meta_dir / "archelect_search.csv" 

print(f"Checking: {meta_file}")

if meta_file.exists():
    print("Found metadata file!")
    # Use low_memory=False to handle the mixed-type warning in Column 10
    master_df = pd.read_csv(meta_file, low_memory=False)
    
    # 1. Parse dates and extract the year
    master_df['parsed_date'] = pd.to_datetime(master_df['date'], errors='coerce')
    
    # 2. Filter out rows with unparseable dates
    master_df = master_df.dropna(subset=['parsed_date']).copy()
    master_df['year'] = master_df['parsed_date'].dt.year.astype(int).astype(str)
    
    # 3. Define the helper function (Indented correctly inside the IF block)
    def fetch_text(manifesto_id, data_dir):
        # Recursively search for the file in all subfolders
        matches = list(data_dir.glob(f"**/{manifesto_id}.txt"))
        if matches:
            try:
                return matches[0].read_text(encoding='utf-8')
            except Exception:
                return None
        return None

    # 4. Fetch the text
    print("Fetching text files... This will take a moment (searching subfolders).")
    # We pass data_dir as an extra argument to the apply function
    master_df['text'] = master_df['id'].apply(fetch_text, data_dir=data_dir)
    
    # 5. Clean-up and Summary
    master_df = master_df.drop(columns=['parsed_date'])
    print(f"Success! Processed DataFrame with {len(master_df)} rows.")
    
    missing_texts = master_df['text'].isna().sum()
    print(f"Found texts for {len(master_df) - missing_texts} out of {len(master_df)} records.")

else:
    print(f"Metadata file not found at: {meta_file.absolute()}")

Checking: /Users/cynthiafrancis/Library/CloudStorage/OneDrive-EcolePolytechnique/Ecole Polytechnique/Year M2/S2/ML for NLP/Project/NLP-Gender-Rhetoric-French-Electoral-Manifestos/metadata/archelect_search.csv
Found metadata file!
Fetching text files... This will take a moment (searching subfolders).
Success! Processed DataFrame with 21194 rows.
Found texts for 21167 out of 21194 records.


We restrict our attention to legislative elections in France from 1973 to 1993. 

In [5]:
# Filter to the specific years of interest
years = ["1973","1978", "1981", "1988", "1993"]
master_df=master_df[master_df['year'].isin(years)]
# Check the distribution of texts across the selected years
year_counts = master_df['year'].value_counts().sort_index()
print("Text counts by year:")
print(year_counts)
# Total number of texts
total_texts = len(master_df)
print(f"Total number of texts in the filtered dataset: {total_texts}")
# Remove rows where 'text' is NaN (i.e., text file was not found or could not be read)
master_df = master_df.dropna(subset=['text'])
print(f"Number of texts after dropping missing ones: {len(master_df)}")

Text counts by year:
year
1973    3843
1978    4830
1981    3133
1988    3551
1993    5837
Name: count, dtype: int64
Total number of texts in the filtered dataset: 21194
Number of texts after dropping missing ones: 21167


# TEXT CLEANING

## Basic normalization

In [6]:
def spacy_compatible_clean(text):
    """
    Lightweight cleaning that preserves accents, apostrophes, and casing 
    so spaCy can accurately process French.
    """
    if not isinstance(text, str):
        return ""
        
    # 1. Remove numbers 
    cleaned_text = re.sub(r'\d+', '', text)
    
    # 2. Collapse multiple spaces, tabs, or newlines into a single space
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

# Apply the lightweight cleaning function
master_df['cleaned_text'] = master_df['text'].apply(spacy_compatible_clean)


# Tokenization

## STOPWORD definition

In [7]:
# Load general grammatical French stopwords
GENERAL_STOPWORDS = set([x.strip().lower() for x in open(data_dir / 'stop_word_fr.txt').readlines()])

# Define domain-specific stopwords
raw_custom_stopwords = {
    "monsieur", "madame","messieurs","mesdames", "mademoiselle", "mesdemoiselles","candidat", "candidate", "candidats","gauche","droite", "élection", 
    "vote", "voter", "circonscription", "département", "république", "sciences", "po", "cevipof", "française", "fonds",
    "liberté", "égalité", "fraternité",  "électeur", "électrice", "électeurs", "électrices", "parti", "partis",
    "janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre", "novembre", "décembre",
    "politique", "france", "français", "française", "françaises", "français", "national", "nationale", "nationaux",
    "élection", "élections", "électoral", "électorale", "électorales", "électoraux", "législative", "législatives", "présidentielle", "présidentielles", "municipale", "municipales",
    "majorité", "minorité", "gouvernement", "opposition", "parlement", "assemblée", "sénat", "député", "députée", "députés", "députées",
    "suppléant", "suppléante", "suppléants", "suppléantes", "programme" , "programmes","commun", "francaise", "francais", 
    "francaises", "action", "moyen", "mesure", "mesures", "projets", "projet", "avenir", "changement", "ensemble", "pays", "citoyen", "citoyenne", 
    "Ain","Aisne","Allier","Alpes-de-Haute-Provence","Hautes-Alpes","Alpes-Maritimes","Ardèche","Ardennes","Ariège","Aube","Aude","Aveyron","Bouches-du-Rhône",
    "Calvados","Cantal","Charente","Charente-Maritime","Cher","Corrèze","Côte-d'Or","Côtes-d'Armor","Creuse","Dordogne","Doubs","Drôme","Eure","Eure-et-Loir",
    "Finistère","Corse-du-Sud","Haute-Corse","Gard","Haute-Garonne","Gers","Gironde","Hérault","Ille-et-Vilaine","Indre","Indre-et-Loire","Isère","Jura","Landes",
    "Loir-et-Cher","Loire","Haute-Loire","Loire-Atlantique","Loiret","Lot","Lot-et-Garonne","Lozère","Maine-et-Loire","Manche","Marne","Haute-Marne","Mayenne",
    "Meurthe-et-Moselle","Meuse","Morbihan","Moselle","Nièvre","Nord","Oise","Orne","Pas-de-Calais","Puy-de-Dôme","Pyrénées-Atlantiques","Hautes-Pyrénées",
    "Pyrénées-Orientales","Bas-Rhin","Haut-Rhin","Rhône","Haute-Saône","Saône-et-Loire","Sarthe","Savoie","Haute-Savoie","Paris","Seine-Maritime","Seine-et-Marne",
    "Yvelines","Deux-Sèvres","Somme","Tarn","Tarn-et-Garonne","Var","Vaucluse","Vendée","Vienne","Haute-Vienne","Vosges","Yonne","Territoire de Belfort","Essonne",
    "Hauts-de-Seine","Seine-Saint-Denis","Val-de-Marne","Val-d'Oise","Guadeloupe","Martinique","Guyane","La Réunion","Mayotte", "mitterrand", "françois", "jean", "lepen", "marine", 
    "social", "sociaux", "sociale", "vie", "pouvoir", "pouvoirs", "peuple"}


# Clean and lemmatize the custom stopwords so they match the text output from spaCy's lemmatization
custom_stopwords_lemmatized = set()

for doc in nlp.pipe(raw_custom_stopwords, disable=['parser', 'ner']):
    for token in doc:
        custom_stopwords_lemmatized.add(token.lemma_.lower())

print(f"Loaded {len(GENERAL_STOPWORDS)} general stopwords.")
print(f"Loaded {len(custom_stopwords_lemmatized)} custom lemmatized stopwords.")

# Define non-lemmatized stopwords for direct addition to the set after lemmatization
raw_stopwords = {"departement", "circonscription", "republique", "republiqu", "liberte", "egalite", "fraternite", 
    "ère", "ere", "ème", "eme", "election", "legislative", "legislatives", "egalité", "sciences", "po", "cevipofscience", "fonds", "président", "république", "michel"
    , "rocard", "bulletin", "degré", "jean", "marie", "pen", "côté", "dimanche", "prochain", "prochaine", "arlette", "laguiller", "grand", "rpr", "udf", "votez", "suppléant",
    "sppléants", "suppléante", "suppléantes"} # I chose these manually based on their frequency in the tokens after 

# Add the additional stopwords to the lemmatized set
custom_stopwords_lemmatized.update(raw_stopwords)


Loaded 701 general stopwords.
Loaded 186 custom lemmatized stopwords.


## Remove general STOPWORDS, lemmatize and POS filter 

In [8]:
ALLOWED_POS = {'NOUN', 'ADJ', 'PROPN'}

def extract_lemmas_base(texts):
    """
    Extracts lemmas, filters POS, and removes ONLY standard grammatical stopwords.
    Leaves domain-specific words intact for n-gram generation.
    """
    lemmatized_list = []
    
    for doc in nlp.pipe(texts, disable=['parser', 'ner']):
        lemmas = [
            token.lemma_.lower() for token in doc 
            if token.is_alpha 
            and not token.is_space                             
            and token.pos_ in ALLOWED_POS                     
            and token.text.lower() not in GENERAL_STOPWORDS           
            and token.lemma_.lower() not in GENERAL_STOPWORDS         
        ]
        lemmatized_list.append(lemmas)
        
    return lemmatized_list

master_df['lemmas_base'] = extract_lemmas_base(master_df['cleaned_text'].astype(str))


# Clean non-alphabetic symbols

In [9]:
import re

VALID_WORD_PATTERN = re.compile(r'^[a-zA-ZàâäéèêëïîôöùûüÿçÀÂÄÉÈÊËÏÎÔÖÙÛÜŸÇ\-]+$')

def remove_symbols_from_tokens(token_list):
    """
    Filters a list of tokens, keeping only pure alphabetical words and hyphenated words.
    """
    if not isinstance(token_list, list):
        return []
        
    cleaned_tokens = []
    for token in token_list:
        # Check if the token perfectly matches our valid characters rule
        if VALID_WORD_PATTERN.match(token) and token != '-':
            cleaned_tokens.append(token)
            
    return cleaned_tokens

# 1. Apply the symbol filter to base lemmas
master_df['lemmas_base_clean'] = master_df['lemmas_base'].apply(remove_symbols_from_tokens)


# N-grams

In [10]:
# Function to apply n-gram generation
def apply_ngrams(tokenized_docs, min_count=5, threshold=10):
    """
    Statistical N-gram Generation using Gensim.
    Stitches frequently co-occurring words together with an underscore.
    """
    
    bigram_model = Phrases(tokenized_docs, min_count=min_count, threshold=threshold)
    
    bigram_phraser = Phraser(bigram_model)
    
    # Apply the bigram model to the documents
    docs_with_bigrams = [bigram_phraser[doc] for doc in tokenized_docs]
    
    return docs_with_bigrams

# (Your apply_ngrams function remains unchanged)
master_df['lemmas_with_ngrams'] = apply_ngrams(master_df['lemmas_base_clean'])


# Custom STOPWORDS removal

In [11]:
def smart_domain_filter(tokenized_docs, custom_stops):
    """
    Robustly filters domain-specific stopwords by checking both individual tokens 
    and the component parts of n-grams (bigrams).
    """
    # 1. Normalize all stopwords into a set for high-speed (O(1)) lookup
    stop_set = {word for word in custom_stops}
    
    cleaned_docs = []
    
    for doc in tokenized_docs:
        doc_clean = []
        for token in doc:
            # 2. Check if the token is an n-gram (contains an underscore)
            if '_' in token:
                parts = token.split('_')
                
                # If EVERY part of the n-gram is in the stop_set, we discard the whole thing
                # (e.g., 'sciences_po' is removed if both 'sciences' and 'po' are stopwords)
                if all(part in stop_set for part in parts):
                    continue  
                else:
                    # Keep the n-gram if at least one part is meaningful
                    doc_clean.append(token)
            
            # 3. Handle single tokens
            else:
                if token not in stop_set:
                    doc_clean.append(token)
                    
        cleaned_docs.append(doc_clean)
        
    return cleaned_docs

# Apply the bulletproof domain filter
master_df['final_tokens'] = smart_domain_filter(
    master_df['lemmas_with_ngrams'], 
    custom_stopwords_lemmatized
)

# Display the first few rows to verify the cleaning results
print(master_df[['cleaned_text', 'final_tokens']].head())

                                        cleaned_text  \
0  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   
1  REPUBLIQUE FRANCAISE - LIBERTE - EGALITE - FRA...   
2  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   
3  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   
4  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   

                                        final_tokens  
0  [paul_barberot, centr, progre_democratie, mode...  
1  [bourg_bresse, union, socialiste_democrate, pa...  
2  [confiance, jeune, chomarat, ecole, livre, lyo...  
3  [fraternité_departement, marcel_benoit, cultiv...  
4  [confiance, lien, sang, amitié, suffrage, étiq...  


# LEAST COMMON TOKENS REMOVAL

In [12]:
# Most common tokens after n-gram generation:
[('travailleur', 26321), ('emploi', 19657), ('union', 18311), ('homme', 17794), ('socialiste', 16314), ('travail', 15436), ('communiste', 15402), ('droit', 15319), ('entreprise', 14886), ('confiance', 13038), ('société', 12882), ('femme', 11989), ('maire', 11877), ('jeune', 11558), ('progrès', 11401), ('région', 11077), ('économique', 10731), ('tour', 10621), ('voix', 9868), ('pourcent', 9664), ('parti_communiste', 9454), ('parti_socialiste', 9445), ('chômage', 9350), ('enfant', 8476), ('service', 8377), ('choix', 8108), ('famille', 7997), ('développement', 7979), ('salaire', 7761), ('lutte', 7619), ('besoin', 7493), ('problème', 7433), ('intérêt', 7409), ('année', 7025), ('public', 6906), ('conseiller_général', 6875), ('volonté', 6800), ('retraite', 6730), ('économie', 6622), ('solidarité', 6590), ('suffrage', 6586), ('défense', 6444), ('europe', 6398), ('temps', 6353), ('place', 6261), ('monde', 6186), ('etat', 6142), ('véritable', 6117), ('formation', 6114), ('démocratie', 6083), ('logement', 6071), ('élu', 5773), ('crise', 5676), ('sécurité', 5673), ('front_national', 5506), ('jour', 5498), ('plan', 5436), ('environnement', 5365), ('ouvrier', 5317), ('ville', 5272), ('commune', 5257), ('agriculteur', 5220), ('justice', 5214), ('conseiller_municipal', 5201), ('loi', 5072), ('victoire', 5064), ('création', 5050), ('nécessaire', 4966), ('aide', 4921), ('centre', 4894), ('agriculture', 4872), ('agricole', 4861), ('activité', 4837), ('local', 4816), ('responsable', 4720), ('condition', 4716), ('général', 4649), ('ancien', 4605), ('soutien', 4547), ('école', 4442), ('responsabilité', 4323), ('ministre', 4292), ('rassemblement', 4243), ('meilleur', 4239), ('actuel', 4227), ('faveur', 4223), ('cadre', 4219), ('populaire', 4213), ('face', 4099), ('espoir', 4075), ('votant', 4067), ('idée', 3976), ('paix', 3965), ('vrai', 3910), ('libre', 3901), ('régional', 3843), ('petit', 3841), ('sens', 3824), ('impôt', 3792), ('priorité', 3784)]
# Least common tokens after n-gram generation:
[('html', 1), ('raou', 1), ('pandraudle', 1), ('raone', 1), ('éver-', 1), ('epi-', 1), ('sodeian', 1), ('résorp-', 1), ('faux-document', 1), ('albretch', 1), ('aseur', 1), ('yna', 1), ('sur-représentée', 1), ('limites', 1), ('sausset', 1), ('bernardelection', 1), ('seine-saint-denispour', 1), ('lotre', 1), ('désiste', 1), ('anti-européen', 1), ('contesté', 1), ('servirait', 1), ('mahèa', 1), ('défosse', 1), ('ipre', 1), ('démocratesmadame', 1), ('nationalepc', 1), ('albertivillarien', 1), ('bourgetin', 1), ('salvator', 1), ('lacquail', 1), ('karmann-', 1), ('gryale', 1), ('votecandidat', 1), ('blanc-mesniloi', 1), ('dugnysiens', 1), ('stanois', 1), ('cafequot', 1), ('treuillois', 1), ('traception', 1), ('apriorisme', 1), ('qaqneron', 1), ('écologauchistes', 1), ('boudy', 1), ('ucho', 1), ('chenevier', 1), ('genle', 1), ('bonl', 1), ('braucent', 1), ('birba', 1), ('batail', 1), ('conne', 1), ('eussir', 1), ('recapitulation', 1), ('pico', 1), ('banfi', 1), ('ladame', 1), ('torisation', 1), ('boissy-saint-légerscience', 1), ('schwartzenking', 1), ('infli-', 1), ('lim', 1), ('segolene', 1), ('kouchnirsciences', 1), ('promue', 1), ('législatives-', 1), ('bebelsky', 1), ('namysl', 1), ('banalisé', 1), ('destabilisé', 1), ('confirmée', 1), ('éliminés', 1), ('éliminé', 1), ('désarmmenent', 1), ('cscianciay', 1), ('charentonscience', 1), ('mauritiens', 1), ('de-marée', 1), ('sionnante', 1), ('divergen-', 1), ('frémainville', 1), ('boise-', 1), ('courdimanche', 1), ('neuville-sur-oise', 1), ('monica', 1), ('tenon', 1), ('incarner', 1), ('-interdire', 1), ('-réexaminer', 1), ('-réclamer', 1), ('vouspanny', 1), ('samelach', 1), ('thatany', 1), ('muontdarfur', 1), ('turi', 1), ('ofcz', 1), ('arnouville-les-gonesse', 1), ('ghettoïsation', 1), ('metais', 1), ('messéant', 1)]

[('html', 1),
 ('raou', 1),
 ('pandraudle', 1),
 ('raone', 1),
 ('éver-', 1),
 ('epi-', 1),
 ('sodeian', 1),
 ('résorp-', 1),
 ('faux-document', 1),
 ('albretch', 1),
 ('aseur', 1),
 ('yna', 1),
 ('sur-représentée', 1),
 ('limites', 1),
 ('sausset', 1),
 ('bernardelection', 1),
 ('seine-saint-denispour', 1),
 ('lotre', 1),
 ('désiste', 1),
 ('anti-européen', 1),
 ('contesté', 1),
 ('servirait', 1),
 ('mahèa', 1),
 ('défosse', 1),
 ('ipre', 1),
 ('démocratesmadame', 1),
 ('nationalepc', 1),
 ('albertivillarien', 1),
 ('bourgetin', 1),
 ('salvator', 1),
 ('lacquail', 1),
 ('karmann-', 1),
 ('gryale', 1),
 ('votecandidat', 1),
 ('blanc-mesniloi', 1),
 ('dugnysiens', 1),
 ('stanois', 1),
 ('cafequot', 1),
 ('treuillois', 1),
 ('traception', 1),
 ('apriorisme', 1),
 ('qaqneron', 1),
 ('écologauchistes', 1),
 ('boudy', 1),
 ('ucho', 1),
 ('chenevier', 1),
 ('genle', 1),
 ('bonl', 1),
 ('braucent', 1),
 ('birba', 1),
 ('batail', 1),
 ('conne', 1),
 ('eussir', 1),
 ('recapitulation', 1),
 ('pi

In [13]:
from collections import Counter
# Remove tokens appearning in less than 5 documents
token_counts = Counter(token for doc in master_df['final_tokens'] for token in set(doc))
tokens_to_keep = {token for token, count in token_counts.items() if count >= 5}
master_df['final_tokens'] = master_df['final_tokens'].apply(lambda doc: [token for token in doc if token in tokens_to_keep])

# SAVE DATASET

In [14]:
# Save the dataframe to a Parquet file 
#!pip install pyarrow

# 1. Convert lists into space-separated strings 
master_df['final_tokens'] = master_df['final_tokens'].apply(
    lambda x: ' '.join(map(str, x)) if isinstance(x, list) else str(x)
)

# 2. Ensure all text columns are standard Python strings 
df_save=master_df.copy()
df_save['identifiant de circonscription'] = df_save['identifiant de circonscription'].astype(str)

# 3. Save to Parquet
df_save.to_parquet('cleaned_master_data.parquet', engine='pyarrow')


In [15]:
# Re-load dataset
master_df = pd.read_parquet('cleaned_master_data.parquet')


In [16]:
master_df

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-soutien,suppleant-liste,suppleant-decorations,year,text,cleaned_text,lemmas_base,lemmas_base_clean,lemmas_with_ngrams,final_tokens
0,EL065_L_1973_03_001_01_1_PF_01,1973-03-04,Assemblée Nationale;France;Ve République;Élect...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,non mentionné,non,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...,"[science, po, fonds, cevipof, republiqu, franç...","[science, po, fonds, cevipof, republiqu, franç...","[science_po, fonds_cevipof, republiqu_français...",centr progre_democratie moderne union_républic...
1,EL065_L_1973_03_001_01_1_PF_02,1973-03-04,Élections législatives;Ve République;Assemblée...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,Parti socialiste;Mouvement des radicaux de gauche,Union de la gauche socialiste et démocrate,non,1973,REPUBLIQUE FRANCAISE - LIBERTE - EGALITE - FRA...,REPUBLIQUE FRANCAISE - LIBERTE - EGALITE - FRA...,"[republique, francaise, liberte, egalite, frat...","[republique, francaise, liberte, egalite, frat...","[republique_francaise, liberte_egalite, frater...",bourg_bresse union socialiste_democrate parti_...
2,EL065_L_1973_03_001_01_1_PF_03,1973-03-04,Assemblée Nationale;Ve République;Élections lé...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,Faîtes confiance aux jeunes d'aujourd'hui,non,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...,"[science, po, fonds, cevipof, republiqu, franç...","[science, po, fonds, cevipof, republiqu, franç...","[science_po, fonds_cevipof, republiqu_français...",confiance jeune ecole livre lyon voyage_étude ...
3,EL065_L_1973_03_001_01_1_PF_04,1973-03-04,France;Ve République;Élections législatives;As...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,Parti communiste français,Union populaire et victoire du programme commun,oui,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...,"[science, po, fonds, cevipof, republiqu, franç...","[science, po, fonds, cevipof, republiqu, franç...","[science_po, fonds_cevipof, republiqu_français...",fraternité_departement cultivateur secrétaire_...
4,EL065_L_1973_03_001_01_1_PF_05,1973-03-04,Élections législatives;France;Assemblée Nation...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,Majorité Ve République,oui,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...,"[science, po, fonds, cevipof, republiqu, franç...","[science, po, fonds, cevipof, republiqu, franç...","[science_po, fonds_cevipof, republiqu_français...",confiance lien sang amitié suffrage étiquette ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21189,EL198_L_1993_03_095_07_2_PF_02,1993-03-28,Ve République;France;Assemblée Nationale;Élect...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,2,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non,1993,Sciences Po / fonds CEVIPOF\nElections législa...,Sciences Po / fonds CEVIPOF Elections législat...,"[science, po, fonds, cevipof, election, législ...","[science, po, fonds, cevipof, election, législ...","[science_po, fonds_cevipof, election_législati...",circonscription_val pluralisme différence conf...
21190,EL198_L_1993_03_095_08_2_PF_01,1993-03-28,Assemblée Nationale;Élections législatives;Fra...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,2,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,Rassemblement pour la République;Union pour la...,non mentionné,non,1993,2e TOUR DES ÉLECTIONS LÉGISLATIVES DU